# Spatial Models Demo: SCL, NestedSCL, MixedSCL, and MixedNestedSCL

This notebook demonstrates the four spatially correlated logit models in `locpick`:

- **SCL** — Spatially Correlated Logit (Bhat & Guo 2004)
- **NestedSCL** — Nested Spatially Correlated Logit
- **MixedSCL** — Mixed Spatially Correlated Logit
- **MixedNestedSCL** — Mixed Nested Spatially Correlated Logit

We use synthetic data generated by `simulate_scl` with a known spatial adjacency structure.

In [ ]:
import numpy as np
import pandas as pd

from locpick import SCL, MixedNestedSCL, MixedSCL, NestedSCL
from locpick.dgp import simulate_scl
from locpick.models.mixed import ParamDistribution
from locpick.models.nested import NestingTree, NestSpec

## 2. Generate Synthetic Data

We use `simulate_scl` to create a synthetic dataset with known spatial correlation. The DGP produces:
- `choosers`: household-level observations with income
- `alternatives`: tract-level attributes (cost, time)
- `adjacency`: a spatial adjacency matrix
- `true_rho`: the ground-truth spatial correlation parameter

In [ ]:
# Generate synthetic SCL data with spatial correlation
n_obs = 2000
n_alts = 20

scl_dataset = simulate_scl(
    n_obs=n_obs,
    n_alts=n_alts,
    alt_params={"cost": -0.5, "time": -0.2},
    rho=0.7,
    seed=42,
)

ct = scl_dataset.choice_table
print(f"Observations: {ct.n_observations}")
print(f"Alternatives: {ct.n_alternatives}")
print(f"True rho: {scl_dataset.true_rho}")
print(f"Adjacency shape: {scl_dataset.adjacency.shape}")

## 3. Configure Spatial Models

We configure four spatial model variants:

1. **SCL** — spatial correlation only
2. **NestedSCL** — spatial + nesting structure
3. **MixedSCL** — spatial + random taste variation
4. **MixedNestedSCL** — spatial + nesting + random variation

Each model shares the same `ChoiceTable` and formula but adds structural complexity. The `graph` parameter accepts a numpy adjacency matrix directly.

In [ ]:
# Common formula for all models
formula = "cost + time - 1"

# Use the adjacency matrix from the DGP directly
adj = scl_dataset.adjacency

# 1. SCL — spatial correlation only
model_scl = SCL(
    ct,
    formula=formula,
    graph=adj,
)

# 2. NestedSCL — spatial + nesting
nest_tree = NestingTree(
    nests=[
        NestSpec(name="urban", alt_ids=list(range(0, n_alts // 2))),
        NestSpec(name="suburban", alt_ids=list(range(n_alts // 2, n_alts))),
    ]
)

model_nested_scl = NestedSCL(
    ct,
    formula=formula,
    graph=adj,
    nests=nest_tree,
)

# 3. MixedSCL — spatial + random coefficients
random_params = {
    "time": ParamDistribution(distribution="normal", param="time"),
}

model_mixed_scl = MixedSCL(
    ct,
    formula=formula,
    graph=adj,
    random_params=random_params,
    n_draws=100,
)

# 4. MixedNestedSCL — spatial + nesting + random coefficients
model_mixed_nested_scl = MixedNestedSCL(
    ct,
    formula=formula,
    graph=adj,
    nests=nest_tree,
    random_params=random_params,
    n_draws=100,
)

print("Models configured:")
print(f"  SCL:            {type(model_scl).__name__}")
print(f"  NestedSCL:      {type(model_nested_scl).__name__}")
print(f"  MixedSCL:       {type(model_mixed_scl).__name__}")
print(f"  MixedNestedSCL: {type(model_mixed_nested_scl).__name__}")

## 4. Fit Spatial Models

Fit each model and inspect the estimated coefficients. The SCL models estimate a `rho` parameter that captures spatial correlation. NestedSCL adds `lambda` nest dissimilarity parameters. MixedSCL adds `sd_*` random coefficient standard deviations.

In [ ]:
# Fit SCL
result_scl = model_scl.fit()
print("=== SCL ===")
print(result_scl.summary())

In [ ]:
# Fit NestedSCL
result_nested_scl = model_nested_scl.fit()
print("=== NestedSCL ===")
print(result_nested_scl.summary())

In [ ]:
# Fit MixedSCL
result_mixed_scl = model_mixed_scl.fit()
print("=== MixedSCL ===")
print(result_mixed_scl.summary())

In [ ]:
# Fit MixedNestedSCL
result_mixed_nested_scl = model_mixed_nested_scl.fit()
print("=== MixedNestedSCL ===")
print(result_mixed_nested_scl.summary())

## 5. Evaluate Model Performance

Compare log-likelihood, AIC, and BIC across the four spatial variants. More complex models should improve fit, but we can check whether the improvement justifies the additional parameters.

In [ ]:
# Collect fit statistics for comparison
results = {
    "SCL": result_scl,
    "NestedSCL": result_nested_scl,
    "MixedSCL": result_mixed_scl,
    "MixedNestedSCL": result_mixed_nested_scl,
}

comparison = pd.DataFrame(
    {
        name: {
            "Log-Likelihood": r.log_likelihood,
            "Null LL": r.log_likelihood_null,
            "AIC": r.aic,
            "BIC": r.bic,
            "Rho²": r.rho_squared,
            "Adj. Rho²": r.rho_bar_squared,
            "n_params": r.n_parameters,
        }
        for name, r in results.items()
    }
).T

print(comparison.round(4))

## 6. Generate Spatial Predictions

Predict choice probabilities for each model. The SCL family produces spatially correlated probabilities — nearby alternatives have more similar predicted shares than under MNL.

In [ ]:
# Predict probabilities using model.probabilities()
# For SCL, pass beta (without rho) and rho separately
beta_scl = result_scl.coefficients.drop("rho").values
rho_scl = result_scl.coefficients["rho"]
probs_scl = model_scl.probabilities(beta=beta_scl, rho=rho_scl)

print(f"SCL probabilities shape: {probs_scl.shape}")
print(f"Probabilities sum to 1: {np.allclose(probs_scl.sum(axis=1), 1.0)}")
print("\nFirst 3 decision-makers' probabilities:")
print(probs_scl[:3].round(4))

## 7. Visualize Spatial Results

Plot the spatial adjacency structure and predicted choice probabilities.

In [ ]:
import matplotlib.pyplot as plt

# Plot 1: Spatial adjacency matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show the adjacency matrix
im = axes[0].imshow(adj[:20, :20], cmap="Blues", interpolation="nearest")
axes[0].set_title("Spatial Adjacency Matrix (first 20 alts)")
axes[0].set_xlabel("Alternative j")
axes[0].set_ylabel("Alternative i")
fig.colorbar(im, ax=axes[0])

# Plot 2: Predicted probabilities for a single decision-maker
axes[1].bar(range(n_alts), probs_scl[0], alpha=0.7, label="SCL")
axes[1].set_xlabel("Alternative")
axes[1].set_ylabel("Predicted Probability")
axes[1].set_title("SCL Predicted Probabilities (DM 0)")
axes[1].legend()

plt.tight_layout()
plt.show()